# Expériences LoRA — The Little Rhyme Maker

Ce notebook déroule toute la partie LoRA du projet : préparation du corpus, entraînement des
deux expériences, évaluation et génération. Il fonctionne **en local** (depuis le dépôt) ou sur
**Google Colab** (Exécution → Modifier le type d'exécution → GPU T4).

Chaque étape appelle un script de `lora_experiments/` : les explications détaillées sont dans
`lora_experiments/README.md`.

In [ ]:
# Mise en place : sur Colab, on clone le dépôt et on installe les dépendances ;
# en local, on se place à la racine du dépôt.
import os, sys
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("Project-LLM").exists():
        !git clone https://github.com/CharlotteHoudaille/Project-LLM.git
    os.chdir("Project-LLM")
    !pip install -q -r lora_experiments/requirements.txt
else:
    root = Path.cwd()
    while not (root / "lora_experiments").exists():
        root = root.parent
    os.chdir(root)
print("dossier :", Path.cwd())

In [ ]:
import torch
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun (CPU, beaucoup plus lent)")

## 1. Le corpus

`prepare_data.py` nettoie les comptines, regroupe les variantes, sépare entraînement / validation /
test par groupe (sans fuite), tire les mots-clés dans chaque comptine et tokenise avec le
tokenizer de SmolLM2. `check_data.py` vérifie le résultat et montre des exemples.

In [ ]:
!{sys.executable} lora_experiments/prepare_data.py
!{sys.executable} lora_experiments/check_data.py --show 2

## 2. LoRA en pratique

On charge SmolLM2-135M, on ajoute les adaptateurs LoRA et on regarde ce qui est entraîné :
seules les petites matrices `lora_A` et `lora_B` ont `requires_grad=True`, et `lora_B` vaut zéro
au départ (le modèle adapté est alors exactement le modèle d'origine).

In [ ]:
import yaml
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM

config = yaml.safe_load(open("lora_experiments/config.yaml", encoding="utf-8"))
model = AutoModelForCausalLM.from_pretrained(config["model"]["name"])
lora = config["lora"]
model = get_peft_model(model, LoraConfig(r=lora["r"], lora_alpha=lora["alpha"], lora_dropout=lora["dropout"],
                                         target_modules=lora["target_modules"], task_type="CAUSAL_LM"))
model.print_trainable_parameters()

layer = model.base_model.model.model.layers[0].self_attn.q_proj
print("poids figé W :", tuple(layer.base_layer.weight.shape), " requires_grad =", layer.base_layer.weight.requires_grad)
print("lora_A       :", tuple(layer.lora_A["default"].weight.shape), " requires_grad =", layer.lora_A["default"].weight.requires_grad)
print("lora_B       :", tuple(layer.lora_B["default"].weight.shape), " vaut zéro au départ :", bool((layer.lora_B["default"].weight == 0).all()))
del model

## 3. L'entraînement

D'abord un test de 10 pas, pour vérifier que tout s'enchaîne. Puis les entraînements complets :
environ 12 à 20 minutes chacun sur une RTX 3050 ou un T4. Les adaptateurs déjà entraînés sont
dans le dépôt (`lora_experiments/outputs/`) : ces cellules sont facultatives.

In [ ]:
!{sys.executable} lora_experiments/train_lora.py --experiment keywords --smoke

In [ ]:
RUN_FULL_TRAINING = False   # passer à True pour réentraîner (≈ 45 min au total)
if RUN_FULL_TRAINING:
    !{sys.executable} lora_experiments/train_lora.py --experiment style
    !{sys.executable} lora_experiments/train_lora.py --experiment keywords
    !{sys.executable} lora_experiments/train_lora.py --experiment keywords --model HuggingFaceTB/SmolLM2-360M --epochs 2 --batch-size 4 --grad-accum 4 --tag _360m

## 4. L'évaluation

Les modèles sont comparés sur les 77 comptines de test : SmolLM2 sans adaptation (135M et 360M),
les deux expériences LoRA et notre modèle historique. Environ 10 à 15 minutes sur GPU.

In [ ]:
RUN_EVALUATION = False      # passer à True pour refaire l'évaluation
if RUN_EVALUATION:
    !{sys.executable} lora_experiments/evaluate.py

In [ ]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display

metrics = json.load(open("lora_experiments/results/metrics.json", encoding="utf-8"))
table = pd.DataFrame(metrics["generation"]).T
display(table.style.format({c: "{:.1%}" for c in table.columns if c not in ("bpc", "perplexity", "words_per_line")}
                           | {"bpc": "{:.2f}", "perplexity": "{:.1f}", "words_per_line": "{:.1f}"}, na_rep="—"))
display(pd.DataFrame(metrics["training"]).T)
display(Image("lora_experiments/results/losses.png"))
display(Image("lora_experiments/results/comparison.png"))

In [ ]:
display(Markdown(open("lora_experiments/results/examples.md", encoding="utf-8").read()))

## 5. Écrire une comptine

Avec l'adaptateur de l'expérience 2 sur SmolLM2-360M.

In [ ]:
!{sys.executable} lora_experiments/generate_lora.py --model keywords_360m --n 3 cat moon